# SN-02 — Construction du périmètre A/B/C

À partir de la sortie SN-01, on récupère les **SIREN validés en Phase 1** (VALIDE_FORT + VALIDE = notre fichier de référence interne).

On construit ensuite le périmètre des EJ à traiter en Phase 2 et 3 :
- **Sous-ensemble A** : EJ rattachés à au moins une EG (via `nmfinessej_stru`)
- **Sous-ensemble B** : EJ non rattachés à une EG mais avec `nmsiren_stru`
- **Sous-ensemble C** : EJ non rattachés à une EG ET sans `nmsiren_stru`

On retire de A et B les EJ dont le SIREN est validé en Phase 1.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.sirenisation import construire_perimetre_abc
from src.display      import afficher_tableau
from config.settings  import (
    FINESS_EJ_CLEAN, FINESS_EG_RAW,
    SN_PHASE1, SN_PERIMETRE, PROCESSED_DIR,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Récupération des SIREN validés en Phase 1

In [2]:
FEUILLES_VALIDES = {'Valide_fort', 'Valide'}

sheets_p1 = pd.read_excel(SN_PHASE1, sheet_name=None, dtype=str)
ids_valides = set()
sirens_valides = set()
for nom, sdf in sheets_p1.items():
    if nom in FEUILLES_VALIDES:
        if 'idstructure_stru' in sdf.columns:
            ids_valides.update(sdf['idstructure_stru'].dropna().astype(str).str.strip())
        if 'nmsiren_stru' in sdf.columns:
            sirens_valides.update(
                sdf['nmsiren_stru'].dropna().astype(str)
                .str.replace(r'\s', '', regex=True).str.strip()
            )
ids_valides.discard('')
sirens_valides.discard('')
print(f'EJ validés en Phase 1 (idstructure)  : {len(ids_valides):,}')
print(f'SIREN validés en Phase 1 (uniques)   : {len(sirens_valides):,}')

EJ validés en Phase 1 (idstructure)  : 38,278
SIREN validés en Phase 1 (uniques)   : 38,154


## 2. Construction du périmètre A/B/C

In [3]:
df_ej = pd.read_parquet(FINESS_EJ_CLEAN)
df_eg = pd.read_parquet(FINESS_EG_RAW)
print(f'EJ FINESS : {len(df_ej):,}')
print(f'EG FINESS : {len(df_eg):,}')

df_perimetre = construire_perimetre_abc(df_ej, df_eg, ids_valides)

n_a = (df_perimetre['sous_ensemble'] == 'A').sum()
n_b = (df_perimetre['sous_ensemble'] == 'B').sum()
n_c = (df_perimetre['sous_ensemble'] == 'C').sum()
print(f'\nEJ à traiter        : {len(df_perimetre):,}')
print(f'  Sous-ensemble A   : {n_a:,}  (rattachés à au moins une EG)')
print(f'  Sous-ensemble B   : {n_b:,}  (sans EG, avec SIREN)')
print(f'  Sous-ensemble C   : {n_c:,}  (sans EG, sans SIREN)')
print(f'EJ exclus (validés en P1) : {len(df_ej) - len(df_perimetre):,}')

EJ FINESS : 54,097
EG FINESS : 104,612

EJ à traiter        : 15,819
  Sous-ensemble A   : 0  (rattachés à au moins une EG)
  Sous-ensemble B   : 13,721  (sans EG, avec SIREN)
  Sous-ensemble C   : 2,098  (sans EG, sans SIREN)
EJ exclus (validés en P1) : 38,278


## 3. Aperçu

In [4]:
afficher_tableau(
    df_perimetre[['idstructure_stru', 'nmsiren_stru', 'raisonsociale_stru',
                  'cdcommune_stru', 'sous_ensemble']],
    f'Aperçu du périmètre A/B/C ({len(df_perimetre):,} lignes)',
)

idstructure_stru,nmsiren_stru,raisonsociale_stru,cdcommune_stru,sous_ensemble
1843328,263003873,CCAS BARJAC,30029,B
1843331,,AAD SOLEIL,30217,C
1843337,,SCM BORRELY CAVALLO DURAND DE LOISON,30189,C
1843339,,ASSOC. SALINDROISE AIDE À DOMICILE,30305,C
1843342,421333667,PHARMACIE LABEILLE,30081,B


## 4. Sauvegarde

In [5]:
df_perimetre.to_parquet(SN_PERIMETRE, index=False)
print(f'Sauvegardé : {SN_PERIMETRE}')

Sauvegardé : /home/jovyan/work/projet_finess_sirene/data/processed/sirenisation_perimetre_abc.parquet
